*Notebook Last Validated: 2026-09-15*

Project UID (Internal, Prod): a8005431-049c-4b00-a5ea-c8d674d507b7

# Hello Numpy Scatter and Gather on Rhino FCP

This notebook walks through running the `hello-numpy-sag` NVFlare example on Rhino's Federated Computing Platform (FCP), end to end, using the Rhino Python SDK.

**What this example actually does:** it's a minimal "hello world" type example for federated learning - there's no real dataset or model. The "model" is just a 3x3 array `[[1,2,3],[4,5,6],[7,8,9]]`, and each training round simply adds 1 to every number in it. The point isn't the math - it's seeing the full federated pipeline (a client and a server talking to each other over several rounds) actually work, before you build something more complex on top of it.

See `README.md` in this directory for the local (non-FCP, Docker-only) walkthrough.

#### Before you run this notebook
1. `pip install rhino_health`
2. Access to a Rhino FCP workgroup and its container registry - see the "Find your workgroup's container registry" section below
3. **Build and push the container image first** (see the next section) - this notebook does not build or push it for you, since that depends on your own registry credentials

## Setup

In [1]:
import json
import os
from getpass import getpass

import rhino_health as rh
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeTypes,
    ModelTrainInput,
)
from rhino_health.lib.endpoints.dataset.dataset_dataclass import DatasetCreateInput
from rhino_health.lib.endpoints.project.project_dataclass import ProjectCreateInput

# Pin the working directory to this notebook's folder so relative paths resolve
# regardless of where the notebook is opened from (e.g. VS Code, Jupyter Lab, etc.)
os.chdir(os.path.dirname(os.path.abspath(globals().get('__vsc_ipynb_file__', 'notebook.ipynb'))))
print("Setup Complete")

Setup Complete


## Find your workgroup's container registry, then build and push the image

> **Do this every time you (re)build the image**, using whatever `app/custom/network.py.enc` / `~/myprecious` the cell above just produced. Reusing an older, already-pushed image - or building before running the cell above - bakes in a mismatched key: the container will crash on decrypt before NVFlare ever starts, which surfaces only as a generic "server and clients failed to connect" timeout with no other error.

FCP runs your own pre-built image rather than building one from source, so you build and push it *outside* this notebook, using the shared push script in `user-resources/utils/`:

1. **Find your workgroup's container repository name** - in the FCP UI, go to **Settings (gear icon) -> Containers & Artifacts -> Workgroup container registry** (there's a copy button next to it).
2. See [Pushing Containers to the ECR](https://docs.rhinofcp.com/getting-started/quick-start-guide/pushing-containers-to-the-ecr) and make sure you've completed all the pre-requisites
3. **Build and push**, from this directory, by entering the following in your command line (replace the elements in <>):
   ```bash
   ../../../utils/docker-push.sh <your-workgroup-repo-name> <a-tag-you-choose>
   ```
   Example: `../../../utils/docker-push.sh workgroup-rhino-health-prod DO-334-test-v1`
   
   This can take some time and will print `Done. Container image URI: ...` when it finishes - **copy that exact URI** into `CONTAINER_IMAGE_URI` below (it's what you actually pushed, so it's more reliable than reconstructing the URI yourself).

In [ ]:
CONTAINER_IMAGE_URI = "865551847959.dkr.ecr.us-east-1.amazonaws.com/workgroup-rhino-health-prod:DO-334-test-v1" # UPDATE THIS TO THE URI PRINTED BY docker-push.sh

if CONTAINER_IMAGE_URI.startswith("<"):
    raise ValueError("Set CONTAINER_IMAGE_URI to the URI printed by docker-push.sh before continuing.")

print("Completed.")

Completed.


## Authentication

Log in to the Rhino FCP. When prompted, provide your password. This proves who you are to the platform - everything you do below (creating a project, registering data, running training) happens under your account.

**Variables to adjust:**
- `USERNAME` - your Rhino FCP username (typically your email)

In [ ]:
USERNAME = "<YOUR_USERNAME>"  # REPLACE WITH YOUR RHINO FCP USERNAME

print("Logging In")
session = rh.login(username=USERNAME, password=getpass())
user = session.current_user
print("Logged In")

Logging In
Logged In


## Project Creation

Everything on FCP - datasets, code, training runs - lives inside a Project. This creates a new one under your primary workgroup to keep this example's resources separate from anything else you're working on.

Note that every time you rerun this cell, it creates a NEW project (a different `project_uid`), even with the same `PROJECT_NAME`.

In [4]:
PROJECT_NAME = "[User-Example] Hello Numpy Scatter and Gather"

project = session.project.add_project(
    ProjectCreateInput(
        name=PROJECT_NAME,
        description="Hello Numpy Scatter and Gather NVFlare demo",
        type="Validation",
        primary_workgroup_uid=user.primary_workgroup_uid,
    )
)
print(f"Created project: {project.name}")

Created project: [User-Example] Hello Numpy Scatter and Gather


## Dataset Creation

FCP runs one federated learning "client" per registered dataset, so training needs at least one dataset to attach to - even though (as noted at the top of this notebook) this example's training code never actually reads anything from it. This registers the small placeholder `data/` folder that ships with this example for that purpose.

`CLIENT_DATA_PATH` must be a path your on-prem client can actually see - swap in a real dataset here once you're building something less trivial than this "hello world" example.

**Variables to adjust:**
- `CLIENT_DATA_PATH` - base path to this repo (or your own data) on your on-prem client's filesystem

In [5]:
CLIENT_DATA_PATH = "/rhino_data/external/import-external-datasets-dev"  # REPLACE WITH YOUR RHINO FCP CLIENT DATA PATH
FILE_BASE_PATH = "user-resource-examples/hello-numpy-sag"  # relative to CLIENT_DATA_PATH
DATASET_NAME = "Hello Numpy SAG Dataset"

dataset = session.dataset.add_dataset(
    DatasetCreateInput(
        name=DATASET_NAME,
        description="Placeholder dataset - not actually used by this example's training code",
        project_uid=project.uid,
        workgroup_uid=project.primary_workgroup_uid,
        file_base_path=os.path.join(CLIENT_DATA_PATH, FILE_BASE_PATH),
        method="filesystem",
        data_schema=None,
        is_data_deidentified=True,
    )
)
print(f"Registered dataset: {dataset.name} ({dataset.uid})")

Registered dataset: Hello Numpy SAG Dataset (b6be1b9f-97ba-4a3d-8047-22e730aa7bd0)


## NVFlare Code Object Creation

A Code Object is FCP's record of "what code to run." This one points at the image you pushed above, rather than having FCP build one from source the way the auto-container examples do.

In [6]:
code_object_input = CodeObjectCreateInput(
    name="Hello Numpy Scatter and Gather",
    description="Hello Numpy Scatter and Gather NVFlare demo",
    input_data_schema_uids=[None],
    output_data_schema_uids=[None],
    project_uid=project.uid,
    code_type=CodeTypes.NVIDIA_FLARE_V2_6,  # must match requirements.txt: nvflare==2.6.0 - FCP's provisioning only supports up to v2.6
    config={"container_image_uri": CONTAINER_IMAGE_URI},
)
# return_existing=False + add_version_if_exists=True: if a Code Object with this name already
# exists (e.g. from a prior run), create a NEW VERSION with this config instead of silently
# returning the old one - otherwise a fixed CONTAINER_IMAGE_URI update above would never
# actually take effect on re-runs.
code_object = session.code_object.create_code_object(
    code_object_input, return_existing=False, add_version_if_exists=True
)
code_object = code_object.wait_for_build()  # returns immediately - the image is already built
print(f"Created code object: {code_object.name} ({code_object.uid})")

Created code object: Hello Numpy Scatter and Gather (79b3f8ee-08ff-483d-9aac-830fb1918661)


In [8]:
# You can run these cells to inspect the code object and its config, and confirm that the container image URI is correct.
co = session.code_object.get_code_object(code_object.uid)
print(co.config)

{'image_tag': 'DO-334-test-v1', 'image_repo_name_part': 'rhino-health-prod', 'container_image_uri': '865551847959.dkr.ecr.us-east-1.amazonaws.com/workgroup-rhino-health-prod:DO-334-test-v1'}


## NVFlare Federated Training Run

Loads the client/server config files, then kicks off training. There's no encryption or secrets in this example, so `secrets_fed_client`/`secrets_fed_server` are left empty. Waits for the run to complete before proceeding.

**Expect to see warnings like this in the logs - they're harmless:**
```
DXOAggregator - WARNING - NUM_STEPS_CURRENT_ROUND missing in meta of DXO from site-1 and set to default value, 1.0.
DXOAggregator - WARNING - Aggregation_weight missing for site-1 and set to default value, 1.0
```
`app/custom/np_trainer.py` returns its result with `meta={}` (empty) - it never reports a step count or aggregation weight, since there's no real training happening to measure. NVFlare's aggregator just falls back to a default of `1.0` for both and warns about it. This is inherited straight from NVIDIA's original example code, unrelated to anything FCP-specific - it shows up every run, at most 10 times per the log message itself, and doesn't affect the result.

In [7]:
with open("app/config/config_fed_client.json") as f:
    config_fed_client = json.dumps(json.load(f))
with open("app/config/config_fed_server.json") as f:
    config_fed_server = json.dumps(json.load(f))

run_params = ModelTrainInput(
    code_object_uid=code_object.uid,
    input_dataset_uids=[dataset.uid],
    one_fl_client_per_dataset=True,
    validation_dataset_uids=[],
    validation_datasets_inference_suffix="",
    timeout_seconds=1200,
    config_fed_client=config_fed_client,
    config_fed_server=config_fed_server,
)

model_train = session.code_object.train_model(run_params)
code_run = model_train.wait_for_completion(1200, poll_frequency=30)
print(f"Training finished with status: {code_run.status}")

Waiting for code run to complete (0 hours 0 minutes and 2 seconds)
Waiting for code run to complete (0 hours 0 minutes and 33 seconds)
Done.
Training finished with status: CodeRunStatus.COMPLETED


In [9]:
# If the training failed, you can inspect the errors to see what went wrong.
print(code_run.errors)

[]


## Model Parameters Download

Downloads the trained model - by now, the numbers should have gone from `[[1,2,3],[4,5,6],[7,8,9]]` to `[[4,5,6],[7,8,9],[10,11,12]]` (one `+1` per round, 3 rounds, per `config_fed_server.json`).

**Note:** you can also download the model parameters directly from the FCP UI by clicking on the three dots next to the code run and selecting "Download Model Paramters.

In [10]:
weights = session.code_run.get_model_params(code_run.uid)
with open("model_parameters.npy", "wb") as f:
    f.write(weights.getbuffer())
print("Saved model to model_parameters.npy")

import numpy as np
print(np.load("model_parameters.npy"))

Saved model to model_parameters.npy
[[ 4.  5.  6.]
 [ 7.  8.  9.]
 [10. 11. 12.]]


## Next Steps: Inference (Local Only)

**"Download" vs. "inference" are two different things, happening in two different places:**
- The **Model Parameters Download** cell above just *fetches the trained model file* (`model_parameters.npy`) from FCP onto your own computer via the SDK - a plain file transfer, no computation involved. After that cell runs, the file sits right here in this notebook's folder.
- **Inference** means actually *using* that model - feeding it some data and getting predictions/scores back out. That's a separate step, and this notebook doesn't do it on FCP at all: this example only trains on FCP, downloads the result, and inference happens **locally**, by running `infer.py` inside the `hello-numpy-sag` Docker container (the same one you built for the "Running this example locally" walkthrough).

(FCP can run inference as its own platform job too, for examples set up to support that - this particular "hello world" example just doesn't wire that path up, so local Docker is the only way to run inference here.)

**To actually try it:** see step 5 of "Running this example locally" in `README.md` - you'll mount this downloaded `model_parameters.npy` (instead of one produced by local training) into the container and run `infer.py` against it there. It needs libraries (`numpy`) that are only installed inside that image, so it won't run directly on your host machine.

**Heads up if you do this:** the `SCORE` column `infer.py` adds will always be `1`, for every row, no matter what. That's not a bug - `infer.py` hardcodes `scores = [1 for row in rows]` rather than doing any real scoring, since (as noted at the top of this notebook) there's no real model here to make predictions with. Swap in real scoring logic once you're adapting this example for an actual use case.

## Cleanup

**On FCP:** this notebook creates a new Project (containing a Dataset and Code Object) every time you run it. Set `CLEANUP = True` below and re-run that cell to remove them once you're done. (This does not delete the pushed container image from your registry.)

**Locally:** see the "Cleanup" section in `README.md` for removing the local Docker image and test directories.

In [ ]:
CLEANUP = False  # Set to True to delete the Project (and its Dataset/Code Object) created above

if CLEANUP:
    session.code_object.remove_code_object(code_object)
    session.dataset.remove_dataset(dataset)
    session.project.remove_project(project)
    print("Removed code object, dataset, and project from FCP.")
else:
    print("Skipped - set CLEANUP = True above to remove the Project/Dataset/Code Object this notebook created.")

## Additional Resources

- [Rhino SDK Documentation](https://rhinohealth.github.io/rhino_sdk_docs/html/autoapi/index.html)
- [Rhino User Resources](https://github.com/RhinoHealth/user-resources/tree/main)
- [Rhino FCP Platform Documentation](https://docs.rhinofcp.com/)
- [NVFlare GitHub](https://github.com/NVIDIA/NVFlare/tree/main)